# Assignment: Barcelona

## Prep

### Imports, shared definitions, datasets

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily
import pointpats
import numpy as np

In [ ]:
def listings(city_path, quarter_end_dates):
    """load listings for a city_path for a series of dates (representing quarter ends) and combines into one DataFrame.

    Adds a new `quarter_end_date` column so that data for each quarter can still be pulled out later.
    """
    listings_url_base = f"https://data.insideairbnb.com/{city_path}"
    dfs = []
    for quarter_end_date in quarter_end_dates:
        listings_url = f"{listings_url_base}/{quarter_end_date}/data/listings.csv.gz"
        df = pd.read_csv(listings_url, compression='gzip')
        df['quarter_end_date'] = quarter_end_date
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [ ]:
listings_df = listings("spain/catalonia/barcelona", ["2024-12-12","2025-03-05","2025-06-12","2025-09-14"])

In [ ]:
listings_df[listings_df['quarter_end_date'] == '2024-12-12']

In [ ]:
listings_df[listings_df['quarter_end_date'] == '2025-09-14']

In [ ]:
listings_geometry = gpd.points_from_xy(listings_df['longitude'], listings_df['latitude'], crs="EPSG:4326")
listings_geometry

In [ ]:
listings_gdf = gpd.GeoDataFrame(listings_df, geometry=listings_geometry)
listings_gdf.head()

In [ ]:
listings_gdf.crs

In [ ]:
listings_gdf.explore(tiles="CartoDB Positron", prefer_canvas=True)

In [ ]:
def price_only(gdf):
    price_gdf = gdf.copy(deep=True)
    price_gdf["price"] = (
        gdf["price"]
          .str.replace(r"[\$,]", "", regex=True)
          .astype(float)
    )
    price_gdf = price_gdf[["price", gdf.geometry.name]]
    price_gdf = price_gdf.dropna()
    return price_gdf
    
listings_price_gdf = price_only(listings_gdf)
listings_price_gdf.head()

In [ ]:
listings_price_gdf.crs

In [ ]:
listings_price_gdf.explore("price", tiles="CartoDB Positron", scheme='percentiles', prefer_canvas=True)

In [ ]:
import h3

def point_to_h3_fn(h3_res):
    """returns a new function which will convert a POINT geometry into an H3 id"""
    def f(row):
        return h3.latlng_to_cell(row.geometry.y, row.geometry.x, h3_res)
    return f

def add_grid_cells(gdf, h3_res):
    """adds a new `h3_id` column to a GDF which is assumed to have a POINT geometry"""
    expected_crs = "EPSG:4326"
    assert gdf.crs == expected_crs, f"needed a CRS of {expected_crs}, but this was {gdf.crs}"
    gdf["h3_id"] = gdf.apply(point_to_h3_fn(h3_res), axis=1)
    
    

In [ ]:
add_grid_cells(listings_price_gdf, h3_res=10)
listings_price_gdf.head()

In [ ]:
from shapely.geometry import Polygon

def h3_to_polygon(h3_id):
    """takes an H3 and returns a boundary as a Shapely Polygon"""
    boundary = h3.cell_to_boundary(h3_id)
    lng_lat = [(lng, lat) for lat, lng in boundary]
    return Polygon(lng_lat)

def visualise_grid_cells(gdf):
    """takes a GDF with an `h3_id` column, or an index, which may contain duplicates, 
    and returns a new GDF with all the unique H3 cells as Polygons"""
    if gdf.index.name == 'h3_id':
        h3_ids = gdf.index
    else:
        h3_ids = gdf['h3_id']
    unique_ids = h3_ids.unique()
    polygon_gdf = gpd.GeoDataFrame(
        {'h3_id': unique_ids},
        geometry=[h3_to_polygon(h) for h in unique_ids],
        crs='EPSG:4326'
    )
    return polygon_gdf
    

In [ ]:
m = visualise_grid_cells(listings_price_gdf).explore(tiles="CartoDB Positron", prefer_canvas=True)
listings_price_gdf.explore("price", m=m, scheme="percentiles", prefer_canvas=True)
m

In [ ]:
# show price distribution

In [ ]:
import seaborn as sns

In [ ]:
sns.displot(listings_price_gdf["price"])

In [ ]:
# the distribution is very skewed so we'll use median rather than mean as a summary of each cell

In [ ]:
median_price_group = listings_price_gdf.groupby(["h3_id"])["price"].median()

In [ ]:
median_price_group

In [ ]:
type(median_price_group)

In [ ]:
def summary_price(gdf):
    """takes a GDF with a 'price' and 'h3_id' column and returns a new GDF with the median price per 'h3_id',
    and a geometry column which is the boundary of the cell as a Polygon"""
    median_price_group = gdf.groupby(["h3_id"])["price"].median()
    summary_gdf = gpd.GeoDataFrame(
        median_price_group,
        geometry=[h3_to_polygon(h) for h in median_price_group.index],
        crs='EPSG:4326'
    )
    return summary_gdf


In [ ]:
listings_price_summary_gdf = summary_price(listings_price_gdf)
listings_price_summary_gdf.head()

In [ ]:
listings_price_summary_gdf.explore("price", tiles="CartoDB Positron", scheme='percentiles')

In [ ]:
m = listings_price_summary_gdf.explore("price", tiles="CartoDB Positron", cmap="Blues", scheme="percentiles", prefer_canvas=True)
listings_price_gdf.explore("price", m=m, cmap="Reds", scheme="percentiles", prefer_canvas=True)
m

In [ ]:
listings_price_gdf.loc[listings_price_gdf["h3_id"] == "8a3944601037fff"]

In [ ]:
listings_price_summary_gdf.loc[listings_price_summary_gdf.index == "8a3944601037fff"]

In [ ]:
m = listings_price_summary_gdf.loc[listings_price_summary_gdf.index == "8a3944601037fff"].explore("price", tiles="CartoDB Positron", scheme="percentiles")
listings_price_gdf.loc[listings_price_gdf["h3_id"] == "8a3944601037fff"].explore("price", m=m, scheme="percentiles")
m

In [ ]:
from libpysal import graph

In [ ]:
listings_price_summary_contiguity = graph.Graph.build_contiguity(listings_price_summary_gdf, rook=False)
listings_price_summary_contiguity

In [ ]:
listings_price_summary_contiguity["8a3944601037fff"]

In [ ]:
type(listings_price_summary_contiguity["8a3944601037fff"])

In [ ]:
listings_price_summary_gdf[listings_price_summary_gdf.index.isin(["8a3944601007fff","8a3944601017fff","8a3944601027fff","8a394460111ffff","8a394460118ffff","8a39446011affff"])].explore()

In [ ]:
listings_price_summary_gdf.index.isin(listings_price_summary_contiguity["8a3944601037fff"])

In [ ]:
m = listings_price_summary_gdf.explore()
listings_price_summary_contiguity.explore(
    listings_price_summary_gdf, m=m, edge_kws=dict(style_kwds=dict(weight=1)), nodes=False
)

In [ ]:
sns.displot(listings_price_summary_contiguity.cardinalities, bins=14, kde=True)

In [ ]:

m = listings_price_summary_gdf.loc[listings_price_summary_contiguity[h3_id].index].explore(color="#25b497")
listings_price_summary_gdf.loc[[h3_id]].explore(m=m, color="#fa94a5")
listings_price_summary_contiguity.explore(listings_price_summary_gdf, m=m, focal=h3_id)

In [ ]:
listings_price_summary_contiguity_r = listings_price_summary_contiguity.transform("r")
listings_price_summary_contiguity_r[h3_id]

In [ ]:
listings_price_summary_gdf["price_lag"] = listings_price_summary_contiguity_r.lag(listings_price_summary_gdf["price"])
listings_price_summary_gdf.head()

In [ ]:
listings_price_summary_gdf.loc[listings_price_summary_contiguity_r[h3_id].index, "price"]

In [ ]:
listings_price_summary_gdf["price_std"] = (
    listings_price_summary_gdf["price"] - listings_price_summary_gdf["price"].mean()
) / listings_price_summary_gdf["price"].std()

In [ ]:
listings_price_summary_gdf.loc[listings_price_summary_contiguity_r[h3_id].index, "price_std"]

In [ ]:
listings_price_summary_gdf["price_std_lag"] = listings_price_summary_contiguity_r.lag(listings_price_summary_gdf["price_std"])

In [ ]:
listings_price_summary_gdf.loc[listings_price_summary_contiguity_r[h3_id].index, "price_std_lag"]

In [ ]:
f, ax = plt.subplots(1, figsize=(6, 6))
sns.regplot(
    x="price_std",
    y="price_std_lag",
    data=listings_price_summary_gdf,
    marker=".",
    scatter_kws={"alpha": 0.2},
    line_kws=dict(color="lightcoral")
)
ax.set_aspect('equal')
plt.axvline(0, c="black", alpha=0.5)
plt.axhline(0, c="black", alpha=0.5)


In [ ]:
import esda

In [ ]:
mi = esda.Moran(listings_price_summary_gdf['price'], listings_price_summary_contiguity_r)

In [ ]:
mi.I

In [ ]:
mi.p_sim

In [ ]:
lisa = esda.Moran_Local(listings_price_summary_gdf['price'], listings_price_summary_contiguity_r)

In [ ]:
listings_price_summary_gdf['cluster'] = lisa.get_cluster_labels(crit_value=0.05)
listings_price_summary_gdf.head()

In [ ]:
lisa.explore(
  listings_price_summary_gdf,
  crit_value=0.05,
  prefer_canvas=True,
  tiles="CartoDB Positron",
)

In [ ]:
_ = lisa.plot_scatter()